# W 보손: 전하 비율 W⁺/W⁻ 와 보이지 않는 중성미자

**CERN 탐구 교실 · 업그레이드 U1 · 노트북 2**

W 보손은 뮤온 하나와 중성미자 하나로 붕괴할 수 있습니다: W⁺ → μ⁺ ν, W⁻ → μ⁻ ν̄.
중성미자는 검출기에 흔적을 남기지 않으므로, **횡운동량 보존**으로 그 존재를 추론합니다. 충돌 전 빔에 수직한 방향(횡방향)의 운동량 합은 0이므로, 보이는 입자들의 횡운동량 합이 0이 아니면 그만큼을 보이지 않는 입자가 가져간 것입니다. 이 양을 **누락 횡에너지 MET** 라고 부릅니다.

| 열 | 뜻 |
|---|---|
| `pt`, `eta`, `phi` | 뮤온의 횡운동량(GeV), 유사 신속도, 방위각(라디안) |
| `Q` | 뮤온 전하 (+1 또는 −1) → W의 전하와 같음 |
| `MET`, `phiMET` | 누락 횡에너지의 크기(GeV)와 방향(라디안) |

**목표**
1. μ⁺ 사건과 μ⁻ 사건을 세어 **W⁺/W⁻ 비**와 통계적 불확도(√N)를 구한다.
2. **횡질량 mT** 분포를 그려 W 질량(80.4 GeV) 근처의 "야코비안 에지"를 찾는다.
3. pt, MET, mT **선택 조건(컷)** 이 결과에 주는 영향을 확인한다.

**데이터 출처**: CMS Open Data (opendata.cern.ch, CC0), 2011년 W → μν 후보 사건 20,000개.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

URL = "https://raw.githubusercontent.com/hkroh99-code/nono99/main/data/cms_wmunu_2011.csv"

def load_csv(url, local_names):
    """인터넷(GitHub)에서 먼저 읽고, 실패하면 내 컴퓨터의 data 폴더에서 읽습니다."""
    try:
        df = pd.read_csv(url)
        print("GitHub에서 읽었습니다:", url)
        return df
    except Exception as err:
        print("인터넷에서 읽지 못했습니다(", type(err).__name__, ") → 로컬 파일을 찾습니다.")
    for path in local_names:
        if os.path.exists(path):
            print("로컬 파일에서 읽었습니다:", path)
            return pd.read_csv(path)
    raise FileNotFoundError("데이터를 찾을 수 없습니다. 인터넷 연결 또는 data 폴더 위치를 확인하세요.")

w = load_csv(URL, ["../data/cms_wmunu_2011.csv", "data/cms_wmunu_2011.csv"])
print("사건 수:", len(w))
w.head()

In [ ]:
w[["pt", "eta", "phi", "Q", "MET", "phiMET"]].describe()

## 1. 전하 세기와 W⁺/W⁻ 비

양성자는 u 쿼크 2개와 d 쿼크 1개(uud)로 이루어져 있습니다. W⁺는 주로 u + d̄ → W⁺, W⁻는 d + ū → W⁻ 과정으로 만들어집니다. 양성자 안에 u가 d보다 많으므로 **W⁺가 더 많이** 만들어질 것으로 예상합니다.

비율 R = N⁺/N⁻ 의 통계적 불확도는 각 개수의 불확도가 √N 이라는 것을 이용해 다음과 같이 구합니다.

$$\sigma_R = R\sqrt{\frac{1}{N^+} + \frac{1}{N^-}}$$

In [ ]:
def charge_ratio(df, label=""):
    n_plus = int((df["Q"] > 0).sum())
    n_minus = int((df["Q"] < 0).sum())
    R = n_plus / n_minus
    dR = R * np.sqrt(1 / n_plus + 1 / n_minus)
    print("%-28s N+ = %5d  N- = %5d  R = %.3f ± %.3f" % (label, n_plus, n_minus, R, dR))
    return n_plus, n_minus, R, dR

charge_ratio(w, "모든 사건(선택 조건 없음)")

## 2. 방위각 차이 Δφ와 횡질량 mT

중성미자의 z방향(빔 방향) 운동량은 알 수 없으므로 W의 불변질량을 완전히 계산할 수는 없습니다. 대신 횡방향 정보만으로 만든 **횡질량**을 씁니다.

$$m_T = \sqrt{2\, p_T\, \mathrm{MET}\, \left(1 - \cos\Delta\phi\right)}$$

Δφ는 뮤온과 MET 방향 사이의 각입니다. 각은 −π~π 범위이므로 차이가 π보다 크면 2π에서 빼서 0~π 범위로 맞춥니다.

mT는 W 질량보다 클 수 없습니다(분해능을 무시하면). 그래서 mT 분포는 80 GeV 근처까지 올라가다가 갑자기 떨어지는 **야코비안 에지(Jacobian edge)** 를 보입니다.

In [ ]:
dphi = np.abs(w["phi"] - w["phiMET"])
dphi = np.where(dphi > np.pi, 2 * np.pi - dphi, dphi)    # 0 ~ π 로 맞추기
w["dphi"] = dphi
w["mT"] = np.sqrt(2 * w["pt"] * w["MET"] * (1 - np.cos(w["dphi"])))

plt.hist(w["mT"], bins=70, range=(0, 140), histtype="stepfilled", color="#4c72b0", alpha=0.8)
plt.axvline(80.4, color="tab:red", linestyle="--", label="W mass 80.4 GeV")
plt.xlabel("Transverse mass mT [GeV]"); plt.ylabel("Events / 2 GeV")
plt.title("W -> mu nu candidates: transverse mass (no cuts)")
plt.legend(); plt.show()

**관찰**: 60~80 GeV에 넓은 봉우리가 있고 80 GeV를 넘으면 빠르게 줄어듭니다. 그런데 mT가 작은 쪽(0~40 GeV)에도 사건이 많습니다. 이 사건들은 대부분 **W가 아닌 배경**(강입자 제트 속 무거운 쿼크가 붕괴해 생긴 뮤온 등)입니다. 이런 사건은 진짜 중성미자가 없어 MET가 작습니다.

## 3. MET 분포 보기

In [ ]:
plt.hist(w["MET"], bins=60, range=(0, 120), histtype="step", linewidth=1.5, color="black")
plt.xlabel("Missing transverse energy MET [GeV]"); plt.ylabel("Events / 2 GeV")
plt.title("MET distribution"); plt.show()

## 4. 선택 조건(컷)의 효과

배경을 줄이려면 MET와 mT에 조건을 겁니다. 조건을 바꿀 때마다 남는 사건 수와 W⁺/W⁻ 비가 어떻게 달라지는지 표로 정리합니다.

In [ ]:
cuts = [
    ("조건 없음",                   0,  0,  0),
    ("MET > 25",                    0, 25,  0),
    ("MET > 25, mT > 50",           0, 25, 50),
    ("pt > 30, MET > 30, mT > 60", 30, 30, 60),
]
rows = []
for name, ptc, metc, mtc in cuts:
    s = w[(w["pt"] > ptc) & (w["MET"] > metc) & (w["mT"] > mtc)]
    n_p, n_m, R, dR = charge_ratio(s, name)
    rows.append({"조건": name, "사건 수": len(s), "N+": n_p, "N-": n_m, "R": round(R, 3), "오차": round(dR, 3)})
pd.DataFrame(rows)

In [ ]:
sel = w[(w["MET"] > 25) & (w["mT"] > 50)]
plt.hist(w["mT"], bins=70, range=(0, 140), histtype="step", color="gray", label="no cuts")
plt.hist(w[w["MET"] > 25]["mT"], bins=70, range=(0, 140), histtype="step", color="black", label="MET > 25 GeV")
plt.hist(sel["mT"], bins=70, range=(0, 140), histtype="stepfilled", alpha=0.5, color="tab:orange", label="MET > 25, mT > 50")
plt.axvline(80.4, color="tab:red", linestyle="--")
plt.xlabel("mT [GeV]"); plt.ylabel("Events / 2 GeV"); plt.legend(); plt.title("Effect of cuts")
plt.show()

**왜 컷을 걸면 R이 커질까?** 배경 사건은 μ⁺와 μ⁻가 거의 같은 수로 나옵니다(R ≈ 1). 배경을 덜어내면 W 사건의 비율이 높아져 R이 W 본래의 값에 가까워집니다. CMS가 7 TeV 충돌에서 측정한 W⁺/W⁻ 생성 비는 약 1.4입니다(검출 범위 보정 후). 우리 값이 조금 다른 이유는 검출 범위(|η| < 2.1)와 효율 보정을 하지 않았고, 배경이 아직 남아 있기 때문입니다.

## 5. μ⁺ 사건과 μ⁻ 사건의 mT 분포 겹쳐 보기

In [ ]:
plt.hist(sel[sel["Q"] > 0]["mT"], bins=45, range=(50, 140), histtype="step", linewidth=1.5, color="tab:red", label="mu+  (W+)")
plt.hist(sel[sel["Q"] < 0]["mT"], bins=45, range=(50, 140), histtype="step", linewidth=1.5, color="tab:blue", label="mu-  (W-)")
plt.xlabel("mT [GeV]"); plt.ylabel("Events / 2 GeV"); plt.legend(); plt.title("W+ vs W- (MET > 25, mT > 50)")
plt.show()

두 분포의 **모양**은 비슷하고(같은 W 질량), **높이**만 다릅니다(생성 개수 차이).

---
## 6. 연습 문제

**연습 1. η 구간별 전하 비.** 뮤온의 |η| 구간을 셋으로 나누어 R을 구하세요. 빔 방향에 가까울수록(|η| 큼) R이 어떻게 변하나요? (양성자 속 u 쿼크가 d 쿼크보다 큰 운동량 비율을 갖는 경우가 많다는 것과 관련 있습니다.)

In [ ]:
eta_edges = None             # 예: [0, 0.8, 1.4, 2.1]

if eta_edges is None:
    print("eta_edges 를 채운 뒤 다시 실행하세요.")
else:
    for lo, hi in zip(eta_edges[:-1], eta_edges[1:]):
        part = sel[(sel["eta"].abs() >= lo) & (sel["eta"].abs() < hi)]
        charge_ratio(part, "%.1f <= |eta| < %.1f" % (lo, hi))

**연습 2. 에지로 W 질량 어림하기.** mT 히스토그램(선택 조건 적용)에서 봉우리 최댓값의 절반이 되는 지점을 **봉우리 오른쪽**에서 찾으세요. 이 값은 W 질량과 얼마나 비슷한가요? 왜 정확히 80.4 GeV가 아닐까요? (분해능, W의 자연 폭, W 자체의 횡운동량을 생각해 보세요.)

In [ ]:
h, e = np.histogram(sel["mT"], bins=np.arange(50, 142, 2))
centers = 0.5 * (e[1:] + e[:-1])
i_peak = int(np.argmax(h))
print("봉우리 위치 ≈ %.0f GeV, 높이 = %d" % (centers[i_peak], h[i_peak]))

half_point = None            # 힌트: 봉우리 오른쪽(centers[i_peak:])에서 h가 h[i_peak]/2 아래로 처음 내려가는 곳
# 예) right = np.where(h[i_peak:] < h[i_peak] / 2)[0][0]; half_point = centers[i_peak + right]

if half_point is None:
    print("half_point 를 구한 뒤 다시 실행하세요.")
else:
    print("반값 지점 ≈ %.0f GeV (W 질량 80.4 GeV)" % half_point)

**연습 3. MET 컷을 바꾸며 R 그래프 그리기.** MET 컷을 0, 10, 20, …, 50 GeV로 바꾸며 R과 오차를 구하고 오차 막대 그래프(`plt.errorbar`)로 그리세요. 컷이 너무 강하면 어떤 문제가 생기나요?

In [ ]:
met_cuts = None              # 예: [0, 10, 20, 30, 40, 50]

if met_cuts is None:
    print("met_cuts 를 채운 뒤 다시 실행하세요.")
else:
    Rs, dRs = [], []
    for c in met_cuts:
        _, _, R, dR = charge_ratio(w[w["MET"] > c], "MET > %d" % c)
        Rs.append(R); dRs.append(dR)
    plt.errorbar(met_cuts, Rs, yerr=dRs, fmt="o-", capsize=4)
    plt.xlabel("MET cut [GeV]"); plt.ylabel("R = N+ / N-"); plt.title("Charge ratio vs MET cut")
    plt.show()

**연습 4 (생각).** 납–납 충돌이나 양성자–반양성자 충돌(예전 미국 테바트론)에서는 W⁺/W⁻ 비가 어떻게 될까요? 양성자–반양성자 충돌에서는 대칭성 때문에 전체 W⁺ 수와 W⁻ 수가 같습니다. 왜 그런지 쿼크 구성으로 설명해 보세요.

## 7. 정리

활동지(U1)에 다음을 기록하세요.
1. 선택 조건 없이 구한 R과 오차, 조건을 건 뒤의 R과 오차
2. mT 분포에서 에지가 나타난 위치와 그 의미
3. 중성미자를 직접 보지 않고도 "있다"고 말할 수 있는 근거
4. 컷을 고를 때 생기는 **절충**(신호 순도 ↔ 남는 사건 수 ↔ 통계 오차)